In [1]:
import os
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/sales.csv'
    OUT_DIR = '/content/drive/MyDrive'
except ImportError:
    DATA_PATH = 'sales.csv'
    OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
def make_demo_sales(n=3636, seed=42):
    rng = np.random.default_rng(seed)
    prices = {'Latte': 38.7, 'Hot Chocolate': 38.7, 'Americano': 28.9,
              'Americano with Milk': 33.8, 'Cappuccino': 38.7, 'Cocoa': 35.76,
              'Cortado': 30.0, 'Espresso': 25.96}
    names = rng.choice(list(prices), size=n)
    start, end = pd.Timestamp('2024-03-01 10:00'), pd.Timestamp('2025-03-23 19:00')
    dt = pd.Series(pd.to_datetime(np.sort(rng.integers(start.value, end.value, size=n))))
    cash_type = np.where(rng.random(n) < 0.9, 'card', 'cash')
    card = [f'ANON-0000-0000-{i:04d}' if t == 'card' else None
            for t, i in zip(cash_type, rng.integers(1, 1200, size=n))]
    return pd.DataFrame({
        'date': dt.dt.strftime('%Y-%m-%d'),
        'datetime': dt.dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3],
        'cash_type': cash_type,
        'card': card,
        'money': [prices[x] for x in names],
        'coffee_name': names,
    })

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    print(f'Файл {DATA_PATH} не знайдено - використовую демонстраційні дані.')
    df = make_demo_sales()

df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 11:58:48.618000,card,ANON-0000-0000-0450,33.80,Hot chocolate
1,2024-03-01,2024-03-01 13:43:22.828000,card,ANON-0000-0000-0207,28.90,Americano
2,2024-03-01,2024-03-01 14:33:30.639000,cash,NaN,35.76,Cocoa
3,2024-03-01,2024-03-01 14:52:46.593000,card,ANON-0000-0000-0484,33.80,Americano with milk
4,2024-03-01,2024-03-01 15:16:40.466000,card,ANON-0000-0000-0285,38.70,Latte


In [5]:
rng = np.random.default_rng(7)
n = 40
df1 = pd.DataFrame({
    'id': range(1, n + 1),
    'name': rng.choice(['Anna', 'Bohdan', 'Olena', 'Taras', 'Iryna'], n),
    'city': rng.choice(['Kyiv', 'Lviv', 'Odesa', 'Kharkiv'], n),
    'age': rng.integers(18, 60, n).astype(float),
    'salary': rng.normal(30000, 5000, n).round(0),
    'score': rng.normal(75, 8, n).round(1),
})

df1.loc[[3, 11, 25], 'age'] = np.nan
df1.loc[[5, 18], 'salary'] = np.nan
df1.loc[[8, 30], 'city'] = None
df1.loc[[2], 'name'] = None
df1.loc[9, 'salary'] = 250000
df1.loc[14, 'age'] = 150
df1.loc[21, 'score'] = -40
df1 = pd.concat([df1, df1.iloc[[0, 1, 4, 10]]], ignore_index=True)
print('Розмір вихідного DataFrame:', df1.shape)
df1.tail(6)

Розмір вихідного DataFrame: (44, 6)


,id,name,city,age,salary,score
38,39,Bohdan,Odesa,34.0,31733.0,72.8
39,40,Iryna,Kyiv,42.0,21559.0,73.7
40,1,Iryna,Lviv,40.0,26622.0,58.7
41,2,Taras,Kyiv,29.0,31016.0,72.6
42,5,Olena,Kharkiv,25.0,24064.0,93.0
43,11,Bohdan,Kyiv,19.0,26027.0,73.4


In [7]:
print(df1.isnull().sum())

id        0
name      1
city      2
age       3
salary    2
score     0
dtype: int64


In [8]:
num_cols = ['age', 'salary', 'score']
cat_cols = ['name', 'city']

df1_clean = df1.copy()
for col in num_cols:
    df1_clean[col] = df1_clean[col].fillna(df1_clean[col].median())
for col in cat_cols:
    df1_clean[col] = df1_clean[col].fillna(df1_clean[col].mode()[0])
print('Пропусків після заповнення:', df1_clean.isnull().sum().sum())
print('Повних дублікатів:', df1_clean.duplicated().sum())
df1_clean = df1_clean.drop_duplicates()
print('Розмір після видалення дублікатів:', df1_clean.shape)

Пропусків після заповнення: 0
Повних дублікатів: 4
Розмір після видалення дублікатів: (40, 6)


In [9]:
mask = pd.Series(True, index=df1_clean.index)
for col in num_cols:
    q1, q3 = df1_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, up = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = (~df1_clean[col].between(low, up)).sum()
    print(f'{col}: межі [{low:.2f}; {up:.2f}], викидів: {n_out}')
    mask &= df1_clean[col].between(low, up)

df1_final = df1_clean[mask].reset_index(drop=True)
print('Розмір після фільтрації викидів:', df1_final.shape)
df1_final.head()

age: межі [16.50; 62.50], викидів: 1
salary: межі [19737.50; 41345.50], викидів: 1
score: межі [52.84; 93.34], викидів: 1
Розмір після фільтрації викидів: (37, 6)


,id,name,city,age,salary,score
0,1,Iryna,Lviv,40.0,26622.0,58.7
1,2,Taras,Kyiv,29.0,31016.0,72.6
2,3,Iryna,Kharkiv,58.0,27683.0,67.8
3,4,Iryna,Kyiv,40.0,30636.0,76.3
4,5,Olena,Kharkiv,25.0,24064.0,93.0


In [10]:
df2 = df.copy()
rng2 = np.random.default_rng(1)


nan_idx = rng2.choice(df2.index, size=5, replace=False)
df2.loc[nan_idx, 'money'] = np.nan


df2 = pd.concat([df2, df2.sample(3, random_state=1)], ignore_index=True)

print('Рядків після додавання дублікатів:', len(df2))
print('Пропусків у money:', df2['money'].isnull().sum())
print('Повних дублікатів:', df2.duplicated().sum())

Рядків після додавання дублікатів: 1003
Пропусків у money: 5
Повних дублікатів: 3


In [11]:
df2 = df2.drop_duplicates()
df2['money'] = df2['money'].fillna(df2['money'].mean())
df2['datetime'] = pd.to_datetime(df2['datetime'], errors='coerce')

print('Рядків після очищення:', len(df2))
print('Пропусків у money:', df2['money'].isnull().sum())
print('Дублікатів:', df2.duplicated().sum())
print(df2.dtypes)

out2 = os.path.join(OUT_DIR, 'cleaned_sales_task2.csv')
df2.to_csv(out2, index=False)
print('Збережено:', out2)

Рядків після очищення: 1000
Пропусків у money: 0
Дублікатів: 0
date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object
Збережено: /content/drive/MyDrive/cleaned_sales_task2.csv


In [12]:
df3 = df.copy()
df3['datetime'] = pd.to_datetime(df3['datetime'], errors='coerce')
df3['year'] = df3['datetime'].dt.year
df3['month'] = df3['datetime'].dt.month
df3['day'] = df3['datetime'].dt.day
df3['day_of_week'] = df3['datetime'].dt.dayofweek   # 0 = понеділок
df3['hour'] = df3['datetime'].dt.hour
df3[['datetime', 'year', 'month', 'day', 'day_of_week', 'hour']].head(10)

,datetime,year,month,day,day_of_week,hour
0,2024-03-01 11:58:48.618,2024,3,1,4,11
1,2024-03-01 13:43:22.828,2024,3,1,4,13
2,2024-03-01 14:33:30.639,2024,3,1,4,14
3,2024-03-01 14:52:46.593,2024,3,1,4,14
4,2024-03-01 15:16:40.466,2024,3,1,4,15
5,2024-03-01 15:23:55.244,2024,3,1,4,15
6,2024-03-01 17:49:57.280,2024,3,1,4,17
7,2024-03-01 17:54:42.744,2024,3,1,4,17
8,2024-03-01 18:26:22.690,2024,3,1,4,18
9,2024-03-01 19:09:35.840,2024,3,1,4,19


In [13]:
df4 = pd.DataFrame({
    'city': ['  Kyiv ', 'LVIV', ' Odesa', 'kharkiv  ', ' DNIPRO '],
    'phone': ['+38 (044) 123-45-67', '+38 (032) 234-56-78', '+38 (048) 345-67-89',
              '+38 (057) 456-78-90', '+38 (056) 567-89-01']
})
print('До очищення:')
print(df4)

df4['city'] = df4['city'].str.strip().str.lower()
df4['phone'] = df4['phone'].str.replace(r'[()\-\s+]', '', regex=True)

print('\nПісля очищення:')
print(df4)

До очищення:
        city                phone
0      Kyiv   +38 (044) 123-45-67
1       LVIV  +38 (032) 234-56-78
2      Odesa  +38 (048) 345-67-89
3  kharkiv    +38 (057) 456-78-90
4    DNIPRO   +38 (056) 567-89-01

Після очищення:
      city         phone
0     kyiv  380441234567
1     lviv  380322345678
2    odesa  380483456789
3  kharkiv  380574567890
4   dnipro  380565678901


In [14]:
df5 = pd.get_dummies(df, columns=['cash_type'], dtype=int)
df5.head()

,date,datetime,card,money,coffee_name,cash_type_card,cash_type_cash
0,2024-03-01,2024-03-01 11:58:48.618000,ANON-0000-0000-0450,33.80,Hot chocolate,1,0
1,2024-03-01,2024-03-01 13:43:22.828000,ANON-0000-0000-0207,28.90,Americano,1,0
2,2024-03-01,2024-03-01 14:33:30.639000,NaN,35.76,Cocoa,0,1
3,2024-03-01,2024-03-01 14:52:46.593000,ANON-0000-0000-0484,33.80,Americano with milk,1,0
4,2024-03-01,2024-03-01 15:16:40.466000,ANON-0000-0000-0285,38.70,Latte,1,0


**Інтерпретація.** Колонка `cash_type` замінена бінарними колонками `cash_type_card`, `cash_type_cash` (за наявності інших значень — відповідно й іншими). У кожному рядку рівно одна з них дорівнює **1** (спосіб оплати цього продажу), решта — **0**. Наприклад, `cash_type_card = 1` означає оплату карткою. Такий формат придатний для алгоритмів машинного навчання, які працюють лише з числами.

In [15]:
df6 = df.copy()
df6['money_norm'] = (df6['money'] - df6['money'].min()) / (df6['money'].max() - df6['money'].min())
df6['money_std'] = (df6['money'] - df6['money'].mean()) / df6['money'].std()

print('money_norm: min =', df6['money_norm'].min(), ', max =', df6['money_norm'].max())
print('money_std: mean =', df6['money_std'].mean(), ', std =', df6['money_std'].std())

print('min == 0:', np.isclose(df6['money_norm'].min(), 0))
print('max == 1:', np.isclose(df6['money_norm'].max(), 1))
print('mean ~ 0:', np.isclose(df6['money_std'].mean(), 0, atol=1e-9))
print('std  ~ 1:', np.isclose(df6['money_std'].std(), 1))
df6[['money', 'money_norm', 'money_std']].head()

money_norm: min = 0.0 , max = 1.0
money_std: mean = 1.2292389328649732e-15 , std = 1.000000000000002
min == 0: True
max == 1: True
mean ~ 0: True
std  ~ 1: True


,money,money_norm,money_std
0,33.80,0.666667,-0.038815
1,28.90,0.333333,-1.065107
2,35.76,0.800000,0.371702
3,33.80,0.666667,-0.038815
4,38.70,1.000000,0.987477


In [16]:
df7 = df.copy()
df7['is_card'] = np.where(df7['cash_type'] == 'card', 1, 0)
df7[['cash_type', 'is_card']].head(10)

,cash_type,is_card
0,card,1
1,card,1
2,cash,0
3,card,1
4,card,1
5,card,1
6,card,1
7,card,1
8,card,1
9,card,1


In [17]:
def clean_dataset(df, date_cols=None):

    out = df.copy()


    out = out.drop_duplicates()

    if date_cols is None:
        date_cols = [c for c in out.columns if 'date' in c.lower() or 'time' in c.lower()]
    for col in date_cols:
        out[col] = pd.to_datetime(out[col], errors='coerce')

    num_cols = out.select_dtypes(include='number').columns
    cat_cols = out.select_dtypes(exclude=['number', 'datetime']).columns
    for col in num_cols:
        out[col] = out[col].fillna(out[col].median())
    for col in cat_cols:
        mode = out[col].mode()
        if not mode.empty:
            out[col] = out[col].fillna(mode.iloc[0])

    mask = pd.Series(True, index=out.index)
    for col in num_cols:
        q1, q3 = out[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        mask &= out[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

    return out[mask].reset_index(drop=True)

In [19]:
rng8 = np.random.default_rng(3)
m = 30
dirty = pd.DataFrame({
    'order_date': pd.date_range('2025-01-01', periods=m, freq='D').strftime('%Y-%m-%d'),
    'product': rng8.choice(['tea', 'coffee', 'cocoa'], m),
    'quantity': rng8.integers(1, 10, m).astype(float),
    'price': rng8.normal(50, 5, m).round(2),
})
dirty.loc[[2, 7], 'quantity'] = np.nan
dirty.loc[[4, 12], 'product'] = None
dirty.loc[10, 'price'] = 900.0
dirty.loc[20, 'quantity'] = 500.0
dirty = pd.concat([dirty, dirty.iloc[[0, 1, 3]]], ignore_index=True)

print('До очищення:', dirty.shape, '| пропусків:', dirty.isnull().sum().sum(),
      '| дублікатів:', dirty.duplicated().sum())

cleaned = clean_dataset(dirty)

print('Після очищення:', cleaned.shape, '| пропусків:', cleaned.isnull().sum().sum(),
      '| дублікатів:', cleaned.duplicated().sum())
print(cleaned.dtypes)
cleaned.head()

До очищення: (33, 4) | пропусків: 4 | дублікатів: 3
Після очищення: (27, 4) | пропусків: 0 | дублікатів: 0
order_date    datetime64[ns]
product               object
quantity             float64
price                float64
dtype: object


,order_date,product,quantity,price
0,2025-01-01,cocoa,7.0,45.57
1,2025-01-02,tea,9.0,48.54
2,2025-01-03,tea,5.0,54.41
3,2025-01-04,tea,3.0,52.90
4,2025-01-05,tea,3.0,50.46


In [20]:
out8 = os.path.join(OUT_DIR, 'cleaned_dataset_task8.csv')
cleaned.to_csv(out8, index=False)
print('Збережено:', out8)

Збережено: /content/drive/MyDrive/cleaned_dataset_task8.csv
